# External Validation — Heart model vs Statlog

> **For research and educational purposes only.** Nothing here is a medical diagnosis or
> medical advice.

Internal test performance says how a model does on held-out rows from the *same* dataset.
External validation asks a harder question: does it hold up on a **different cohort**?

This notebook reads the artifacts written by

```
python -m ml.external.heart_statlog
```

**The result is a negative one, and that is the finding.** Statlog turned out not to be an
independent cohort at all, so the heart model has no external validation. The check that
established this is reusable and should be run against any future candidate dataset.


In [1]:
import sys, os
_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if _root not in sys.path:
    sys.path.insert(0, _root)
os.chdir(_root)

import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 60)

ext = json.loads(Path("reports/heart/external_validation.json").read_text(encoding="utf-8"))
print("status                        :", ext["status"])
print("schema compatible             :", ext["schema_compatible"])
print("usable as external validation :", ext["usable_as_external_validation"])


status                        : rejected
schema compatible             : True
usable as external validation : False


## 1. Encoding compatibility — the assumption carried since Phase 1

Aligning a second dataset by column *name* is only safe if the *encodings* agree. A `thal`
coded 3/6/7 in one release and 0/1/2 in another aligns silently and predicts nonsense.

This was an untested assumption until now. It **holds** — but verifying it matters for a
second reason: it is what makes the contamination finding below meaningful. The rows really
are the same records, not coincidentally equal under a mismatched encoding.


In [2]:
display(pd.DataFrame(ext["schema_check"]))

,column,in_both,values_a,values_b,compatible,note
0,age,True,"[29, 77] mean 54.44","[29, 77] mean 54.43",True,
1,trestbps,True,"[94, 200] mean 131.69","[94, 200] mean 131.34",True,
2,chol,True,"[126, 564] mean 246.69","[126, 564] mean 249.66",True,
3,thalach,True,"[71, 202] mean 149.61","[71, 202] mean 149.68",True,
4,oldpeak,True,"[0, 6.2] mean 1.04","[0, 6.2] mean 1.05",True,
5,cp,True,"1, 2, 3, 4","1, 2, 3, 4",True,
6,restecg,True,"0, 1, 2","0, 1, 2",True,
7,slope,True,"1, 2, 3","1, 2, 3",True,
8,ca,True,"0, 1, 2, 3","0, 1, 2, 3",True,
9,thal,True,"3, 6, 7","3, 6, 7",True,


## 2. Is the dataset actually independent?

Measured **before** any metric is computed. Matching is exact on all 13 features after
numeric normalisation — casting to float first, because the same record arriving as `int64`
in one release and `float64` in another would otherwise never match, silently reporting a
contaminated dataset as clean.


In [3]:
ov = ext["independence_check"]
display(pd.Series({k: v for k, v in ov.items() if not isinstance(v, list)}).to_frame("value"))
display(Markdown("**Reasons:**\n" + "\n".join(f"- {r.capitalize()}." for r in ov["reasons"])))


,value
n_candidate,270
n_reference,303
n_matched,270
pct_matched,100.0
n_label_agreements,270
n_unseen,0
pct_unseen,0.0
n_in_train,222
n_in_test,48
pct_in_train,82.22


**Reasons:**
- 270 of 270 rows (100.0%) match a row in the training dataset exactly on every feature, against a maximum acceptable overlap of 5%.
- All 270 matched rows also carry the same label, so these are the same records rather than coincidentally similar patients.
- 222 rows (82.22%) are in the model's own training split — the model was fitted on them.
- No row in this dataset is unseen by the model.

In [4]:
display(Markdown(f"> {ext['verdict']}"))

> NOT INDEPENDENT — this dataset is a subset of the training dataset, not an independent cohort. 270 of 270 rows (100.0%) match a row in the training dataset exactly on every feature, against a maximum acceptable overlap of 5%. All 270 matched rows also carry the same label, so these are the same records rather than coincidentally similar patients. 222 rows (82.22%) are in the model's own training split — the model was fitted on them. No row in this dataset is unseen by the model. Any metric computed on it measures memorisation as much as generalisation and must not be published as external validation.

## 3. The metrics — recorded, not to be cited

The model was scored on Statlog anyway. Read the number, then read what it is worth.

This is the part worth dwelling on: the contaminated result does **not** look broken. It
lands just *below* the internal test score, which is exactly the modest confirmation a
sound external validation would produce. A fake that looks obviously wrong is harmless;
this one would have passed review.


In [5]:
display(pd.Series(ext["metrics"]).to_frame("value"))
display(Markdown(f"> {ext['metrics_interpretation']}"))


,value
n,270.000000
prevalence,0.444444
threshold,0.500000
accuracy,0.866667
precision,0.862069
recall_sensitivity,0.833333
specificity,0.893333
f1,0.847458
roc_auc,0.939944
pr_auc,0.935323


> THESE NUMBERS ARE NOT EXTERNAL VALIDATION. ROC-AUC 0.9399 on n=270 was computed on rows the model was largely trained on (222/270 are training rows, 0 are unseen). They are recorded only to show how plausible a contaminated result looks: it sits just below the internal test score, which is what a sound external validation would also produce. Do not cite them.

In [6]:
internal = json.loads(Path("reports/heart/metrics.json").read_text(encoding="utf-8"))
compare = pd.DataFrame({
    "internal test (honest, n=61)": internal["test_set_threshold_0.5"],
    "statlog (CONTAMINATED, n=270)": ext["metrics"],
}).loc[["n", "roc_auc", "pr_auc", "recall_sensitivity", "specificity", "precision", "accuracy"]]
display(compare)
print("The two columns are close. That closeness is the trap, not the reassurance.")


,"internal test (honest, n=61)","statlog (CONTAMINATED, n=270)"
n,61.000000,270.000000
roc_auc,0.953463,0.939944
pr_auc,0.949561,0.935323
recall_sensitivity,0.892857,0.833333
specificity,0.878788,0.893333
precision,0.862069,0.862069
accuracy,0.885246,0.866667


The two columns are close. That closeness is the trap, not the reassurance.


## 4. What this means

The heart model has **no external validation**. Its only honest performance estimate stays
the 61-row held-out Cleveland test set, with every limitation already in the model card:
a single-site, referral-based cohort from the late 1980s, far too small for narrow
confidence intervals.

Genuine external validation would need a heart cohort not derived from the Cleveland
database — the Hungarian, Switzerland or Long Beach partitions distributed alongside it are
candidates, and each would have to pass this same independence check first.


In [7]:
display(Markdown(Path("reports/heart/EXTERNAL_VALIDATION.md").read_text(encoding="utf-8")))

# External validation — heart model vs Statlog (UCI id 145)

**Status: REJECTED** — this dataset cannot be used to validate the heart model.

## Verdict

> NOT INDEPENDENT — this dataset is a subset of the training dataset, not an independent cohort. 270 of 270 rows (100.0%) match a row in the training dataset exactly on every feature, against a maximum acceptable overlap of 5%. All 270 matched rows also carry the same label, so these are the same records rather than coincidentally similar patients. 222 rows (82.22%) are in the model's own training split — the model was fitted on them. No row in this dataset is unseen by the model. Any metric computed on it measures memorisation as much as generalisation and must not be published as external validation.

## 1. Encoding compatibility (the Phase 1 assumption)

Carried since Phase 1: that `cp`, `slope` and `thal` use the same codes in the Statlog release as in the Cleveland processed release. Aligning columns by name without checking this would align silently and predict nonsense.

**Result: compatible.**

| column | in_both | values_a | values_b | compatible | note |
|---|---|---|---|---|---|
| age | True | [29, 77] mean 54.44 | [29, 77] mean 54.43 | True |  |
| trestbps | True | [94, 200] mean 131.69 | [94, 200] mean 131.34 | True |  |
| chol | True | [126, 564] mean 246.69 | [126, 564] mean 249.66 | True |  |
| thalach | True | [71, 202] mean 149.61 | [71, 202] mean 149.68 | True |  |
| oldpeak | True | [0, 6.2] mean 1.04 | [0, 6.2] mean 1.05 | True |  |
| cp | True | 1, 2, 3, 4 | 1, 2, 3, 4 | True |  |
| restecg | True | 0, 1, 2 | 0, 1, 2 | True |  |
| slope | True | 1, 2, 3 | 1, 2, 3 | True |  |
| ca | True | 0, 1, 2, 3 | 0, 1, 2, 3 | True |  |
| thal | True | 3, 6, 7 | 3, 6, 7 | True |  |
| sex | True | 0, 1 | 0, 1 | True |  |
| fbs | True | 0, 1 | 0, 1 | True |  |
| exang | True | 0, 1 | 0, 1 | True |  |

## 2. Dataset independence

External validation only means anything if the model has not seen the rows. Matching is exact on all 13 features after numeric normalisation.

| check | value |
|---|---|
| `n_candidate` | 270 |
| `n_reference` | 303 |
| `n_matched` | 270 |
| `pct_matched` | 100.0000 |
| `n_label_agreements` | 270 |
| `n_unseen` | 0 |
| `pct_unseen` | 0.0000 |
| `n_in_train` | 222 |
| `n_in_test` | 48 |
| `pct_in_train` | 82.2200 |
| `independent` | False |

Reasons this dataset is not independent:

- 270 of 270 rows (100.0%) match a row in the training dataset exactly on every feature, against a maximum acceptable overlap of 5%.
- All 270 matched rows also carry the same label, so these are the same records rather than coincidentally similar patients.
- 222 rows (82.22%) are in the model's own training split — the model was fitted on them.
- No row in this dataset is unseen by the model.

## 3. Metrics — recorded, not to be cited

> THESE NUMBERS ARE NOT EXTERNAL VALIDATION. ROC-AUC 0.9399 on n=270 was computed on rows the model was largely trained on (222/270 are training rows, 0 are unseen). They are recorded only to show how plausible a contaminated result looks: it sits just below the internal test score, which is what a sound external validation would also produce. Do not cite them.

| metric | value |
|---|---|
| `n` | 270 |
| `prevalence` | 0.4444 |
| `threshold` | 0.5000 |
| `accuracy` | 0.8667 |
| `precision` | 0.8621 |
| `recall_sensitivity` | 0.8333 |
| `specificity` | 0.8933 |
| `f1` | 0.8475 |
| `roc_auc` | 0.9399 |
| `pr_auc` | 0.9353 |
| `brier` | 0.0964 |
| `cm_tn` | 134 |
| `cm_fp` | 16 |
| `cm_fn` | 20 |
| `cm_tp` | 100 |

## What this means for the heart model

The heart model has **no external validation**. Its only honest performance estimate remains the 61-row held-out Cleveland test set, with all the limitations already recorded in the model card: a single-site, referral-based cohort from the late 1980s, far too small for narrow confidence intervals.

This is a negative result and it is the correct one. Publishing the Statlog numbers would have manufactured false evidence of generalisation, and because they land just below the internal test score they would not have looked wrong.

Genuine external validation would need a heart cohort that is not derived from the Cleveland database — for example the Hungarian, Switzerland or Long Beach partitions distributed alongside it, each of which would first have to pass this same independence check.

---
*Generated by `ml.external.heart_statlog.run_external_validation`. Every number above comes from that run.*
